In [1]:
# Import necessary libraries and modules for data analysis
import glob  # For file path matching
import os  # For interacting with the operating system
import numpy as np  # For numerical operations and array handling
import pandas as pd
from astropy.io import ascii
import matplotlib.pyplot as plt  # For data visualization
import math  # For mathematical functions
from scipy.optimize import curve_fit  # For fitting curves to data
from scipy.spatial.distance import pdist, squareform  # For distance calculations and transforming data
from scipy.signal import argrelextrema  # For finding relative extrema in data
from scipy.ndimage import gaussian_filter  # For applying a Gaussian filter to data
from scipy.stats import spearmanr, pearsonr
from matplotlib.lines import Line2D  # Import Line2D
from fractions import Fraction
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.gridspec as gridspec
from tqdm import tqdm  # For displaying progress bars
from astropy.table import Table

# Get the default color cycle for plotting
color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [2]:
################################################################################
# Read mFISH Data
################################################################################

def READ_NPY(filename, dir):
    """
    Read pairwise distances matrix from an NPY file.

    Parameters:
    - filename (str): The name of the NPY file.
    - dir (str): The directory containing the file.

    Returns:
    - numpy.ndarray: Pairwise distances matrix.
    """
    file = os.path.join(dir, filename)
    distances_matrix = np.load(file)
    return distances_matrix

def READ_CSV(filename, dir):
    """
    Read mFISH data from a CSV file.

    Parameters:
    - filename (str): The name of the CSV file.
    - dir (str): The directory containing the file.

    Returns:
    - numpy.ndarray: Numpy array containing data in micrometers.
    """
    file_path = f'{dir}/{filename}'
    lines = [ln[:-1].split(',') for ln in open(file_path, 'r')]
    data = []
    for line in lines[1:]:
        if len(line) > 1:
            row_data = [float(val) if val else np.nan for val in line[2:]]
            data.append(row_data)
    data_array = np.array(data)
    print("Shape of data_array:", data_array.shape)
    nchr = len(np.unique(data_array[:, 0]))
    print("Number of chromosomes:", nchr)
    print("Expected size for reshaping:", nchr, -1, 3)
    chrZXY = data_array[:, 1:].reshape(nchr, -1, 3)
    return chrZXY


def NonNanPaths(zxys):
    """
    Filter out paths with NaN values.

    Parameters:
    - zxys (numpy.ndarray): 3D array containing data for multiple cells.

    Returns:
    - numpy.ndarray: Filtered cells containing non-NaN values.
    - list: List of zxys arrays for non-NaN cells.
    """
    cells = np.array([c for c in range(len(zxys)) if not True in np.isnan(zxys[c])], dtype=int)
    zxys_ = []
    for c in cells:
        zxys_.append(zxys[c])
    return cells, zxys_


In [3]:
import os
import numpy as np
import pandas as pd
from astropy.io import ascii

def convert_barcode(segment_number):
    barcode_mapping = {
        1: 1, 2: 2, 3: 3, 7: 4, 8: 5, 9: 6, 10: 7, 11: 8, 12: 9, 13: 10,
        14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16, 20: 17, 21: 18,
        23: 19, 24: 20, 26: 21, 27: 22, 28: 23, 639: 24, 708: 25
    }
    return barcode_mapping.get(segment_number, None)

def READ_mFISH_ECSV(filename, dir):
    try:
        # Construct the file path
        file_path = os.path.join(dir, filename)
        print("File path:", file_path)  # Print file path for debugging

        # Check if the file exists
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"No such file or directory: '{file_path}'")

        # Read the ECSV file using astropy
        data_table = ascii.read(file_path, format='ecsv')

        # Convert the data to a pandas DataFrame
        df = data_table.to_pandas()

        # Sort DataFrame by Trace_ID
        df.sort_values(by='Trace_ID', inplace=True)

        # Initialize variables
        current_trace_id = None
        chromosome_index = 0

        # Iterate over DataFrame rows to assign Chromosome index and Segment index
        for index, row in df.iterrows():
            if row['Trace_ID'] != current_trace_id:
                chromosome_index += 1
                current_trace_id = row['Trace_ID']
            df.at[index, 'Chromosome index'] = chromosome_index
            try:
                df.at[index, 'Segment index'] = convert_barcode(row['Barcode #'])
            except Exception as e:
                print(f"Error processing row {index}: {e}")

        # Reorder columns
        df = df[['Chromosome index', 'Segment index', 'z', 'x', 'y']]

        # Rename columns to match desired output
        df.rename(columns={'z': 'Z', 'x': 'X', 'y': 'Y'}, inplace=True)

        # Fill missing segment indexes with NaN and ensure each chromosome index has 25 segment indexes
        complete_df = pd.DataFrame()
        for chromosome_index, group in df.groupby('Chromosome index'):
            missing_segment_indexes = set(range(1, 26)).difference(group['Segment index'])
            missing_data = pd.DataFrame({
                'Chromosome index': [chromosome_index] * len(missing_segment_indexes),
                'Segment index': list(missing_segment_indexes),
                'Z': [np.nan] * len(missing_segment_indexes),
                'X': [np.nan] * len(missing_segment_indexes),
                'Y': [np.nan] * len(missing_segment_indexes)
            })
            complete_df = pd.concat([complete_df, group, missing_data], ignore_index=True)

        # Sort by Chromosome index and then by Segment index
        complete_df.sort_values(by=['Chromosome index', 'Segment index'], inplace=True)

        # Fill empty x, y, and z values with NaN
        complete_df[['X', 'Y', 'Z']] = complete_df[['X', 'Y', 'Z']].fillna(np.nan)

        # Extracting the file name from the directory path
        text = filename + 'rut'
        csv_file_name = text + '_xyzs.csv'

        # Save DataFrame to CSV
        output_file = os.path.join(dir, csv_file_name)
        # Save DataFrame to CSV with NaN representation
        complete_df.to_csv(output_file, index=False, na_rep='NaN')

        print("CSV file saved successfully:", output_file)

        return output_file

    except Exception as e:
        print("Error:", e)



In [5]:
# Main code
rut_directories = '/Users/Loucif/Desktop/PhD/Olivier/'

rut_cell_lines = [ 'Non-Kenyon all', 'Non-Kenyon AB', 'Non-Kenyon ABp', 'Non-Kenyon G']
    #, 'Kenyon', r"$\alpha \beta$ Kenyon", r"$\alpha' \beta'$ Kenyon", r"$\gamma$ Kenyon"]

rut_file_names = ['combined_all_traces_not_KC_all.ecsv', 'combined_all_traces_not_KC_AB.ecsv', 'combined_all_traces_not_KC_ABp.ecsv', 'combined_all_traces_not_KC_G.ecsv']
    #, 'combined_all_traces_KC_all.ecsv', 'combined_all_traces_KC_AB.ecsv', 'combined_all_traces_KC_ABp.ecsv', 'combined_all_traces_KC_G.ecsv']

rut_colors = ['k','gray','tab:orange',
              'tab:brown',
              'tab:red']

rut_npy_files = ['Matrx_not_KCs.npy']

rut_bc_type = [1,2, r"$E_{0a}$",r"$E_{0b}$", r"$E_{1}$", r"$E_{1}E_{2}$", r"$E_{2}$", 8,9, 10,  r"$P_\text{rut}$",12,13,14, 15, 16,17,18,19,20,21,22,23,24,25]
rut_bc_type_color = ['k', 'k', 'lightcoral', 'lightcoral', 'lightcoral', 'lightcoral', 'lightcoral', 'k', 'k', 'k', 'seagreen', 'seagreen', 'k', 'k', 'k', 'k', 'k', 'k', 'k', 'k', 'k', 'k', 'k', 'k', 'k']

rut_TAD_border = 14
file_path = '/Users/remini/Desktop/PhD/olivier/Library_Summary_rut_4RT.txt'
rut_pos = [int((int(line.split()[1]) + int(line.split()[2])) / 2) for line in open(file_path, 'r')]
rut_ss = [0] + [rut_pos[i] - rut_pos[i-1] for i in range(1, len(rut_pos))]

# Create dictionaries to store results
sNPF_RESULTS = {}
rut_RESULTS = {}

for directory, cell_line, file_name,  color in zip([rut_directories]*len(rut_cell_lines), rut_cell_lines, rut_file_names, rut_colors):
    rut_RESULTS[cell_line] = {}
    rut_RESULTS[cell_line]['name'] = cell_line
    rut_RESULTS[cell_line]['zxys'] = READ_mFISH_ECSV(filename=file_name, dir=directory)

print("done")

File path: /Users/Loucif/Desktop/PhD/Olivier/combined_all_traces_not_KC_all.ecsv
Error: No such file or directory: '/Users/Loucif/Desktop/PhD/Olivier/combined_all_traces_not_KC_all.ecsv'
File path: /Users/Loucif/Desktop/PhD/Olivier/combined_all_traces_not_KC_AB.ecsv
Error: No such file or directory: '/Users/Loucif/Desktop/PhD/Olivier/combined_all_traces_not_KC_AB.ecsv'
File path: /Users/Loucif/Desktop/PhD/Olivier/combined_all_traces_not_KC_ABp.ecsv
Error: No such file or directory: '/Users/Loucif/Desktop/PhD/Olivier/combined_all_traces_not_KC_ABp.ecsv'
File path: /Users/Loucif/Desktop/PhD/Olivier/combined_all_traces_not_KC_G.ecsv
Error: No such file or directory: '/Users/Loucif/Desktop/PhD/Olivier/combined_all_traces_not_KC_G.ecsv'
done


In [6]:
import pandas as pd

# Define the file paths
file_path = '/Users/Loucif/Desktop/PhD/Olivier/combined_all_labels_not_KCs_fixed_with_index.ecsv'
output_file_path = '/Users/Loucif/Desktop/PhD/Olivier/mod_combined_all_traces_KC_not.ecsv'

# Read the ECSV file into a DataFrame, skipping bad lines
df = pd.read_csv(file_path, comment='#', delimiter=r'\s+')

# Fill the Chrom column with 'xxxxx'
df['Chrom'] = 'xxxxx'

# Reorder the columns to place Chrom before Chrom_Start
columns = list(df.columns)
if 'Chrom_Start' in columns:
    columns.insert(columns.index('Chrom_Start'), columns.pop(columns.index('Chrom')))
    df = df[columns]
else:
    print("Column 'Chrom_Start' not found in the DataFrame.")

# Save the modified DataFrame back to an ECSV file with the original header
with open(file_path, 'r') as f:
    header = ''.join([next(f) for _ in range(17)])  # Read the first 17 lines (header)

with open(output_file_path, 'w') as f:
    f.write(header)
    df.to_csv(f, index=False, sep=' ')

print(f"Modified ECSV file saved successfully: {output_file_path}")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/Loucif/Desktop/PhD/Olivier/combined_all_labels_not_KCs_fixed_with_index.ecsv'

In [7]:
import os
import numpy as np
import pandas as pd
from astropy.io import ascii

def convert_barcode(segment_number):
    barcode_mapping = {
        1: 1, 2: 2, 3: 3, 7: 4, 8: 5, 9: 6, 10: 7, 11: 8, 12: 9, 13: 10,
        14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16, 20: 17, 21: 18,
        23: 19, 24: 20, 26: 21, 27: 22, 28: 23, 639: 24, 708: 25
    }
    return barcode_mapping.get(segment_number, None)

def READ_mFISH_ECSV(filename, dir):
    try:
        # Construct the file path
        file_path = os.path.join(dir, filename)
        print("File path:", file_path)  # Print file path for debugging

        # Check if the file exists
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"No such file or directory: '{file_path}'")

        # Read the ECSV file using astropy
        data_table = ascii.read(file_path, format='ecsv')

        # Convert the data to a pandas DataFrame
        df = data_table.to_pandas()

        # Print the columns of the DataFrame for debugging
        print("Columns in the DataFrame:", df.columns)

        # Add missing columns with NaN values if they are not present
        required_columns = ['Spot_ID', 'Trace_ID', 'x', 'y', 'z',  'Chrom_Start', 'Chrom_End', 'ROI #', 'Mask_id', 'Barcode #', 'label']
        for col in required_columns:
            if col not in df.columns:
                df[col] = np.nan

        # Select the desired columns
        df = df[required_columns]

        # Sort DataFrame by Trace_ID
        df.sort_values(by='Trace_ID', inplace=True)

        # Initialize variables
        current_trace_id = None
        chromosome_index = 0

        # Iterate over DataFrame rows to assign Chromosome index and Segment index
        for index, row in df.iterrows():
            if row['Trace_ID'] != current_trace_id:
                chromosome_index += 1
                current_trace_id = row['Trace_ID']
            df.at[index, 'Chromosome index'] = chromosome_index
            try:
                df.at[index, 'Segment index'] = convert_barcode(row['Barcode #'])
            except Exception as e:
                print(f"Error processing row {index}: {e}")

        # Reorder columns
        df = df[['Chromosome index', 'Segment index', 'z', 'x', 'y']]

        # Rename columns to match desired output
        df.rename(columns={'z': 'Z', 'x': 'X', 'y': 'Y'}, inplace=True)

        # Fill missing segment indexes with NaN and ensure each chromosome index has 25 segment indexes
        complete_df = pd.DataFrame()
        for chromosome_index, group in df.groupby('Chromosome index'):
            missing_segment_indexes = set(range(1, 26)).difference(group['Segment index'])
            missing_data = pd.DataFrame({
                'Chromosome index': [chromosome_index] * len(missing_segment_indexes),
                'Segment index': list(missing_segment_indexes),
                'Z': [np.nan] * len(missing_segment_indexes),
                'X': [np.nan] * len(missing_segment_indexes),
                'Y': [np.nan] * len(missing_segment_indexes)
            })
            complete_df = pd.concat([complete_df, group, missing_data], ignore_index=True)

        # Sort by Chromosome index and then by Segment index
        complete_df.sort_values(by=['Chromosome index', 'Segment index'], inplace=True)

        # Fill empty x, y, and z values with NaN
        complete_df[['X', 'Y', 'Z']] = complete_df[['X', 'Y', 'Z']].fillna(np.nan)

        # Extracting the file name from the directory path
        text = filename + 'rut'
        csv_file_name = text + '_xyzs.csv'

        # Save DataFrame to CSV
        output_file = os.path.join(dir, csv_file_name)
        # Save DataFrame to CSV with NaN representation
        complete_df.to_csv(output_file, index=False, na_rep='NaN')

        print("CSV file saved successfully:", output_file)

        return output_file

    except Exception as e:
        print("Error:", e)

In [ ]:
# Main code
rut_directories = '/Users/Loucif/Desktop/PhD/Olivier/'

READ_mFISH_ECSV(filename='combined_all_traces_KC_not.ecsv', dir=rut_directories)

print("done")

In [ ]:
import pandas as pd

# Step 1: Read the CSV file into a DataFrame, skipping bad lines
file_path = '/Users/Loucif/Desktop/PhD/Olivier/combined_all_labels_not_KCs.ecsv'
df = pd.read_csv(file_path, error_bad_lines=False)

# Step 2: Convert columns to numeric, coercing errors to NaN
df[['X', 'Y', 'Z']] = df[['X', 'Y', 'Z']].apply(pd.to_numeric, errors='coerce')

# Step 3: Identify rows with NaN values
non_numeric_rows = df[df[['X', 'Y', 'Z']].isna().any(axis=1)]

# Print the rows with non-numeric values
print("Rows with non-numeric values:")
print(non_numeric_rows)

# Step 4: Fix or remove the non-numeric values
# For example, you can remove these rows
df_cleaned = df.dropna(subset=['X', 'Y', 'Z'])

# Save the cleaned DataFrame to a new CSV file
cleaned_file_path = '/Users/Loucif/Desktop/PhD/Olivier/cleaned_combined_all_labels_not_KCs.csv'
df_cleaned.to_csv(cleaned_file_path, index=False)

In [ ]:
import os

def compare_file_headers(file1, file2):
    with open(file1, 'r') as f1, open(file2, 'r') as f2:
        header1 = f1.readline().strip()
        header2 = f2.readline().strip()
        print(f"Header of {file1}: {header1}")
        print(f"Header of {file2}: {header2}")
        if header1 == header2:
            print("Headers are the same.")
        else:
            print("Headers are different.")

def compare_file_samples(file1, file2, num_lines=17):
    with open(file1, 'r') as f1, open(file2, 'r') as f2:
        print(f"First {num_lines} lines of {file1}:")
        for _ in range(num_lines):
            print(f1.readline().strip())
        print(f"\nFirst {num_lines} lines of {file2}:")
        for _ in range(num_lines):
            print(f2.readline().strip())

# Paths to the files
file1 = '/Users/Loucif/Desktop/PhD/Olivier/combined_all_labels_not_KCs.ecsv'
file2 = '/Users/Loucif/Desktop/PhD/Olivier/combined_all_traces_KC_all.ecsv'

# Compare headers
compare_file_headers(file1, file2)

# Compare first few lines
compare_file_samples(file1, file2)